# Tutorial 5: PyMC Surrogate Learning (`pymc_gp`)

Estimated time: 30-50 minutes

## Prerequisites
`pymc` and `arviz` installed.

## Learning aims
- Primary package aim: fit/evaluate surrogate models through the CLI and inspect artifacts
- Secondary scientific aim: build intuition for prior, posterior, and posterior predictive uncertainty

## Success criteria
- you can train a PyMC surrogate, evaluate new inputs, and interpret uncertainty width


## Why this tutorial matters
Each step connects the CLI workflow to scientific reasoning so you can explain not only *what* ran, but *why* results are meaningful.


## Step 1: Ensure training dataset exists


In [ ]:
%%bash
cd /Users/barak/Downloads/metamodeler_codex_scaffold_docs
PYTHONPATH=src python -m metamodeler.cli.main run tutorials/specs/model.toy.grid.json


## Step 2: Fit surrogate and list artifacts


In [ ]:
%%bash
cd /Users/barak/Downloads/metamodeler_codex_scaffold_docs
PYTHONPATH=src python -m metamodeler.cli.main surrogate fit tutorials/specs/surrogate.toy.pymc_gp.json
PYTHONPATH=src python -m metamodeler.cli.main surrogate list


## Step 3: Evaluate on new inputs


In [ ]:
%%bash
cd /Users/barak/Downloads/metamodeler_codex_scaffold_docs
PYTHONPATH=src python -m metamodeler.cli.main surrogate eval   tutorials/specs/surrogate.toy.pymc_gp.json   --inputs '{"a":[0.25,0.75,1.25,1.75],"b":[0.2,0.6,1.0,1.4]}'   --n 200


## Step 4: Plot predictive mean and uncertainty (graphic)


In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

from metamodeler.spec import SurrogateSpec
from metamodeler.surrogates import eval_surrogate

spec_payload = json.loads(Path('/Users/barak/Downloads/metamodeler_codex_scaffold_docs/tutorials/specs/surrogate.toy.pymc_gp.json').read_text())
spec = SurrogateSpec.model_validate(spec_payload)
inputs = {"a": [0.25, 0.75, 1.25, 1.75], "b": [0.2, 0.6, 1.0, 1.4]}
result = eval_surrogate(spec=spec, inputs_payload=inputs, n=300)

mean = np.asarray(result['summary']['mean'], dtype=float)
std = np.asarray(result['summary'].get('std', [0.0] * len(mean)), dtype=float)
x = np.arange(len(mean))

plt.figure(figsize=(6, 4))
plt.errorbar(x, mean, yerr=std, fmt='o-', capsize=4)
plt.title('PyMC surrogate predictive mean ± std')
plt.xlabel('query point index')
plt.ylabel('predicted y')
plt.grid(True, alpha=0.3)
plt.show()


## Scientific mini-lesson
- Prior: beliefs before data.
- Posterior: updated beliefs after data.
- Posterior predictive: uncertainty-aware output prediction.

Interpretation prompt:
- Where are uncertainties widest, and what does that say about data support in that region?


In [ ]:
%%bash
cd /Users/barak/Downloads/metamodeler_codex_scaffold_docs
pytest -q tests/test_surrogate_backends.py -k pymc_gp_backend_fit_sample_and_logprob
